In [1]:
import numpy as np
import matplotlib.pyplot as plt

In [12]:
def save_and_display_image(input_bin_file, output_txt_file, width, height, mode=0):

    # Read all bytes from the binary file
    with open(input_bin_file, "rb") as file:
        byte_values = list(file.read())        
    # Save byte values to a text file
    with open(output_txt_file, "w") as file:
        file.write(" ".join(map(str, byte_values)))

    if mode == 0:
        # Extract every second byte because each pixel is stored in a 16-bit word
        pixel_values = byte_values[::2]

        if len(pixel_values) != width * height:
            side = int(len(pixel_values) ** 0.5)
            image = np.array(pixel_values[: side * side]).reshape((side, side))
        else:
            image = np.array(pixel_values).reshape((height, width))

    elif mode == 1:
        # Combine pairs of bytes into signed 16-bit values (Little Endian)
        pixel_values_16bit = []

        for i in range(0, len(byte_values) - 1, 2):
            low_byte = byte_values[i]
            high_byte = byte_values[i + 1]

            unsigned_value = low_byte | (high_byte << 8)
            signed_value = np.int16(unsigned_value - 65536) if unsigned_value > 32767 else np.int16(unsigned_value)

            pixel_values_16bit.append(signed_value)

        # Convert fixed-point values (Q9.7) to floating-point
        pixel_values = [value / 128 for value in pixel_values_16bit]

        if len(pixel_values) < width * height:
            height = len(pixel_values) // width

        image = np.array(pixel_values[: width * height]).reshape((height, width))

    else:
        raise ValueError("Mode must be either 0 or 1.")

    # Display the image
    plt.figure(figsize=(10, 8))
    plt.imshow(image, cmap="gray", vmin=0, vmax=255)
    plt.axis("off")





def save_and_display_original_image(filename, output_txt_file, width, height):
    
    # Avoid first 8 bytes (header)
    img = np.fromfile(filename, dtype=np.uint8, offset=8)

    # Check size
    if img.size != width * height:
        raise ValueError(
            f"Fajl sadrži {img.size} bajtova slike, "
            f"a očekivano je {width * height}."
        )

    # Save byte values to a text file
    with open(output_txt_file, "w") as f:
        f.write(" ".join(map(str, img.tolist())))

    # Create picture
    img = img.reshape((height, width))

    # Display the image
    plt.figure(figsize=(10, 8))
    plt.imshow(img, cmap="gray", vmin=0, vmax=255)
    plt.axis("off")


In [13]:
"""
save_and_display_image("ulazna_slika.bin", "izlazna_slika.txt", sirina-(maska-1), visina-(maska-1), mode 0 ili 1)

#PRIMER 1: (128)
ulazna slika: lena_128.bin
velicina: 128x128
mod: 0
maska: 3x3

save_and_display_image("lena_128.bin", "lena_128.txt", 126, 126, mode=0)
plt.savefig("lena_128.png")

#PRIMER 2: (512)
ulazna slika: lena_512.bin
velicina: 512x512
mod: 1
maska: 7x7

save_and_display_image("lena_512.bin", "lena_512.txt", 506, 506, mode=1)
plt.savefig("lena_512.png")

#PRIMER 3: (bypass)
ulazna slika: lena_128.bin
bypass: 1

save_and_display_image("bypass128.bin", "bypass128.txt", 128, 128, mode=0)
plt.savefig("bypass128.png")

#PRIMER 4: (original)
ulazna slika: lena_128.bin

save_and_display_original_image("lena_128.bin","lena_128.txt",128,128)
plt.savefig("lena_128.png")
""";